# Calculate Regrowth Fractions for Antibiotic Gradient Experiment

## Overview
This notebook calculates regrowth fractions across spatial gradients for the antibiotic gradient experiment and generates intermediate CSV files for downstream analysis and plotting.

## Analysis Workflow
1. **Define Spatial Bins**: Create 10 × 10 grid of bins spanning 0-50 µm in x and y directions
2. **Calculate Regrowth Fractions**: Normalize colony counts by mean cell density (`b_mean`)
3. **Aggregate Across Levels**: 
   - Per chamber: Individual chamber-bin combinations
   - Per replicate: Average across chambers within each replicate
   - Final: Average across all replicates

## Outputs
- `1_first_colonies_ab_gradient_merged.csv`: Combined colony data from all replicates
- `1_regrowth_fractions_ab_gradient_per_chamber.csv`: Regrowth fractions for each chamber-bin
- `1_regrowth_fractions_ab_gradient_per_replicate.csv`: Regrowth fractions averaged per replicate
- `1_regrowth_fractions_ab_gradient_merged.csv`: Final regrowth fractions with statistics across replicates

In [1]:
# Data analysis packages
import numpy as np
import pandas as pd

# File handling
import os

# Verify working directory
print(f"Current working directory: {os.getcwd()}")
print(f"Expected to be in: .../Figure2/S9_ab_gradient/figure_code/")

Current working directory: /Users/simonvanvliet/Library/CloudStorage/Dropbox/Work/Code/Spatial-Tolerance-Figure-Data-and-Code/FigureS9_ab_gradient/figure_code
Expected to be in: .../Figure2/S9_ab_gradient/figure_code/


# Step 1: Define Spatial Bins for Colony Analysis

## Objective
Create a uniform 10 × 10 grid of spatial bins for analyzing colony positions.

## Configuration
- Number of bins: 10 in each dimension (x and y)
- Spatial range: 0-50 µm
- Bin width: 5 µm
- Normalization factor (b_mean): 148.46 cells per bin (same as wild-type)

## Output
- `x_bins`: Array of bin edges along x-axis
- `y_bins`: Array of bin edges along y-axis

In [2]:
# Spatial binning parameters
pixel_to_um = 0.065  # Pixel size conversion factor
num_bins = 10  # Number of bins per axis
max_distance = 50  # Maximum distance in micrometers

# Normalization factor (mean cells per bin, same as wild-type)
b_mean = 159.659649122807

# Create bin edges (evenly spaced from 0 to 50 µm)
x_bins = np.linspace(0, max_distance, num_bins + 1)
y_bins = np.linspace(0, max_distance, num_bins + 1)

# Calculate bin width
bin_width = (x_bins[1] - x_bins[0])

# Print configuration
print(f"Spatial binning configuration:")
print(f"  Number of bins per axis: {num_bins}")
print(f"  Spatial range: 0-{max_distance} µm")
print(f"  Bin width: {bin_width:.2f} µm")
print(f"  Normalization factor (b_mean): {b_mean:.2f} cells per bin")
print(f"\nBin edges (µm):")
print(f"  x_bins: {x_bins}")
print(f"  y_bins: {y_bins}")

Spatial binning configuration:
  Number of bins per axis: 10
  Spatial range: 0-50 µm
  Bin width: 5.00 µm
  Normalization factor (b_mean): 159.66 cells per bin

Bin edges (µm):
  x_bins: [ 0.  5. 10. 15. 20. 25. 30. 35. 40. 45. 50.]
  y_bins: [ 0.  5. 10. 15. 20. 25. 30. 35. 40. 45. 50.]


# Step 2: Calculate Regrowth Fractions

## Objective
Compute regrowth fractions for each spatial bin by normalizing colony counts with the mean cell density (`b_mean`).

## Workflow
1. **Load Colony Data**: Import first-colony detection data from the replicate
2. **Filter Colonies**: 
   - Remove merged colonies (ID ≥ 10,000)
   - Restrict to first 20 hours of observation
3. **Spatial Binning**: Assign each colony to x and y bins
4. **Count Colonies**: Tabulate colonies per chamber per bin (including chambers with zero regrowth)
5. **Calculate Regrowth Fractions**: Normalize by `b_mean` at three levels:
   - **Per chamber**: Individual regrowth fraction for each chamber-bin combination
   - **Per replicate**: Average across all chambers within a replicate (accounting for chambers with zero regrowth)
   - **Final**: Average across all replicates with standard deviation

## Normalization Formula
$$\text{Regrowth Fraction} = \frac{\text{Colony Count}}{b_{\text{mean}}}$$

Where `b_mean` is the average number of cells per bin from the initial cell distribution.

## Outputs
- Per-chamber regrowth fractions
- Per-replicate regrowth fractions
- Merged regrowth fractions with statistics

In [3]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Define replicate data sources (relative paths to analysis outputs)
replicates = [
    {
        'colonies_path': '../analysis_code/2_first_colonies_ab_gradient.csv',
        'replicate': 'rep1'
    }
]

# Define output file paths
OUTPUT_COLONIES = '1_first_colonies_ab_gradient_merged.csv'
OUTPUT_REGROWTH_FINAL = '1_regrowth_fractions_ab_gradient_merged.csv'
OUTPUT_PER_REP_COMBINED = '1_regrowth_fractions_ab_gradient_per_replicate.csv'
OUTPUT_PER_CHAMBER = '1_regrowth_fractions_ab_gradient_per_chamber.csv'

print(f"Normalization factor (b_mean): {b_mean:.2f}\n")

# =============================================================================
# STEP 1: Compute Per-Chamber Regrowth Fractions
# =============================================================================

all_colonies = []
regrowth_per_chamber = []
total_chambers_per_replicate = {}

for rep in replicates:
    path = rep["colonies_path"]
    rep_name = rep["replicate"]

    # Check if file exists
    if not os.path.exists(path):
        print(f"⚠ Skipping {rep_name} (file not found: {path})")
        continue

    # Load colony data
    df = pd.read_csv(path)
    
    # Get ALL chambers (not just those with regrowth)
    all_positions = sorted(df["position"].dropna().unique())
    total_chambers = len(all_positions)
    total_chambers_per_replicate[rep_name] = total_chambers

    print(f"{rep_name}: {total_chambers} chambers analyzed")
    print(f"  Positions: {', '.join(all_positions)}")
    print("-" * 80)

    # Filter to valid colonies
    colonies = df[df["colony_id"].notna()].copy()
    
    if colonies.empty:
        print(f"  {rep_name}: No regrowth detected\n")
        positions_with_colonies = []
    else:
        # Convert frames to hours and apply time/ID filters
        colonies["hours"] = colonies["frame"] * (5 / 60)  # 5 min/frame
        colonies = colonies[(colonies["hours"] <= 20) & (colonies["colony_id"] < 10000)].copy()
        
        if not colonies.empty:
            # Add metadata and convert to micrometers
            colonies["replicate"] = rep_name
            colonies["x_um"] = colonies["x"] * pixel_to_um
            colonies["y_um"] = colonies["y"] * pixel_to_um

            # Assign colonies to spatial bins
            colonies["x_bin"] = pd.cut(colonies["x_um"], bins=x_bins, include_lowest=True)
            colonies["y_bin"] = pd.cut(colonies["y_um"], bins=y_bins, include_lowest=True)
            
            all_colonies.append(colonies)
            positions_with_colonies = colonies["position"].unique()
            
            print(f"  {rep_name}: {len(colonies)} colonies after filtering")
        else:
            print(f"  {rep_name}: No colonies after filtering")
            positions_with_colonies = []

    # Create complete chamber-bin template for ALL chambers (including zeros)
    bin_lefts_y = [y_bins[i] for i in range(len(y_bins)-1)]
    bin_lefts_x = [x_bins[i] for i in range(len(x_bins)-1)]
    
    # Create template for Y-axis: all positions × all bins
    template_y = pd.DataFrame([
        {'position': pos, 'bin_left': bin_left, 'axis': 'y'}
        for pos in all_positions
        for bin_left in bin_lefts_y
    ])
    
    # Create template for X-axis: all positions × all bins
    template_x = pd.DataFrame([
        {'position': pos, 'bin_left': bin_left, 'axis': 'x'}
        for pos in all_positions
        for bin_left in bin_lefts_x
    ])
    
    # --- Count colonies per chamber per bin (Y-axis) ---
    if not colonies.empty:
        chamber_bin_counts_y = (
            colonies.groupby(["position", "y_bin"])
            .size()
            .reset_index(name="colony_count")
        )
        chamber_bin_counts_y["bin_left"] = chamber_bin_counts_y["y_bin"].apply(
            lambda x: float(x.left) if pd.notna(x) else None
        )
        chamber_bin_counts_y = chamber_bin_counts_y.dropna(subset=["bin_left"])
        chamber_bin_counts_y["bin_left"] = chamber_bin_counts_y["bin_left"].astype(float).clip(lower=0)
        chamber_bin_counts_y = chamber_bin_counts_y[["position", "bin_left", "colony_count"]]
    else:
        chamber_bin_counts_y = pd.DataFrame(columns=["position", "bin_left", "colony_count"])
    
    # Merge with template to include zeros
    chamber_y_complete = template_y.merge(
        chamber_bin_counts_y, 
        on=["position", "bin_left"], 
        how="left"
    ).fillna(0)
    chamber_y_complete["replicate"] = rep_name
    chamber_y_complete["colony_count"] = chamber_y_complete["colony_count"].astype(int)
    chamber_y_complete["regrowth_fraction"] = chamber_y_complete["colony_count"] / b_mean
    
    # --- Count colonies per chamber per bin (X-axis) ---
    if not colonies.empty:
        chamber_bin_counts_x = (
            colonies.groupby(["position", "x_bin"])
            .size()
            .reset_index(name="colony_count")
        )
        chamber_bin_counts_x["bin_left"] = chamber_bin_counts_x["x_bin"].apply(
            lambda x: float(x.left) if pd.notna(x) else None
        )
        chamber_bin_counts_x = chamber_bin_counts_x.dropna(subset=["bin_left"])
        chamber_bin_counts_x["bin_left"] = chamber_bin_counts_x["bin_left"].astype(float).clip(lower=0)
        chamber_bin_counts_x = chamber_bin_counts_x[["position", "bin_left", "colony_count"]]
    else:
        chamber_bin_counts_x = pd.DataFrame(columns=["position", "bin_left", "colony_count"])
    
    # Merge with template to include zeros
    chamber_x_complete = template_x.merge(
        chamber_bin_counts_x, 
        on=["position", "bin_left"], 
        how="left"
    ).fillna(0)
    chamber_x_complete["replicate"] = rep_name
    chamber_x_complete["colony_count"] = chamber_x_complete["colony_count"].astype(int)
    chamber_x_complete["regrowth_fraction"] = chamber_x_complete["colony_count"] / b_mean
    
    # Store per-chamber data (now includes zeros)
    regrowth_per_chamber.append(
        chamber_y_complete[["replicate", "position", "axis", "bin_left", "colony_count", "regrowth_fraction"]]
    )
    regrowth_per_chamber.append(
        chamber_x_complete[["replicate", "position", "axis", "bin_left", "colony_count", "regrowth_fraction"]]
    )

# Combine per-chamber data from all replicates
if regrowth_per_chamber:
    regrowth_per_chamber_df = pd.concat(regrowth_per_chamber, ignore_index=True)
else:
    regrowth_per_chamber_df = pd.DataFrame(
        columns=["replicate", "position", "axis", "bin_left", "colony_count", "regrowth_fraction"]
    )

# Save merged colonies data
merged_colonies_df = pd.concat(all_colonies, ignore_index=True) if all_colonies else pd.DataFrame()
merged_colonies_df.to_csv(OUTPUT_COLONIES, index=False)
print(f"\n✅ Saved merged colonies to: {OUTPUT_COLONIES}")

# Save per-chamber regrowth data
regrowth_per_chamber_df.to_csv(OUTPUT_PER_CHAMBER, index=False)
print(f"✅ Saved per-chamber regrowth fractions to: {OUTPUT_PER_CHAMBER}")

# =============================================================================
# STEP 2: Normalize Per Replicate (Include Chambers with Zero Regrowth)
# =============================================================================

# Sum regrowth fractions within each replicate-axis-bin combination
replicate_bin_stats = (
    regrowth_per_chamber_df
    .groupby(["replicate", "axis", "bin_left"], as_index=False)
    .agg(total_fraction_in_bin=("regrowth_fraction", "sum"))
)

# Normalize by total number of chambers (including those with zero regrowth)
replicate_bin_stats["total_chambers"] = replicate_bin_stats["replicate"].map(total_chambers_per_replicate)
replicate_bin_stats["regrowth_fraction"] = (
    replicate_bin_stats["total_fraction_in_bin"] / replicate_bin_stats["total_chambers"]
)

# Save per-replicate regrowth data
replicate_bin_stats[["replicate", "axis", "bin_left", "regrowth_fraction"]].to_csv(
    OUTPUT_PER_REP_COMBINED, index=False
)
print(f"✅ Saved per-replicate regrowth fractions to: {OUTPUT_PER_REP_COMBINED}")

# =============================================================================
# STEP 3: Average Across Replicates
# =============================================================================

n_replicates = len(replicates)

# Calculate mean and standard deviation across replicates
bin_stats = (
    replicate_bin_stats
    .groupby(["axis", "bin_left"], as_index=False)
    .agg(
        mean_regrowth_fraction=("regrowth_fraction", lambda x: x.sum() / n_replicates),
        std_regrowth_fraction=("regrowth_fraction", "std"),
        n_replicates=("replicate", "nunique")
    )
)

# Fill NaN standard deviations with zero (for bins with only one replicate)
bin_stats["std_regrowth_fraction"] = bin_stats["std_regrowth_fraction"].fillna(0)

# Save final merged regrowth data
bin_stats.to_csv(OUTPUT_REGROWTH_FINAL, index=False)
print(f"✅ Saved final regrowth fractions to: {OUTPUT_REGROWTH_FINAL}")

# =============================================================================
# PRINT SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("REGROWTH FRACTION PER BIN (mean ± std across replicates)")
print("=" * 80)

# Print results for each axis
for axis in ["x", "y"]:
    print(f"\n{axis.upper()}-AXIS:")
    print("-" * 80)
    axis_stats = bin_stats[bin_stats["axis"] == axis].sort_values("bin_left")

    for _, row in axis_stats.iterrows():
        b0, b1 = row["bin_left"], row["bin_left"] + bin_width
        print(f"  Bin {b0:5.1f}-{b1:5.1f} µm: "
              f"{row['mean_regrowth_fraction']:.6f} ± {row['std_regrowth_fraction']:.6f} "
              f"(n={row['n_replicates']})")

    # Calculate overall statistics for this axis
    overall_mean = axis_stats["mean_regrowth_fraction"].mean()
    overall_std = axis_stats["mean_regrowth_fraction"].std()
    print(f"\n  Overall {axis}-axis mean: {overall_mean:.6f} ± {overall_std:.6f}")

# Print global summary
print("\n" + "=" * 80)
print("OVERALL SUMMARY")
print("=" * 80)
print(f"Total colonies detected: {len(merged_colonies_df)}")
print(f"Total chamber-bin combinations: {len(regrowth_per_chamber_df)}")
print(f"Mean regrowth fraction (all bins): "
      f"{bin_stats['mean_regrowth_fraction'].mean():.6f} ± "
      f"{bin_stats['std_regrowth_fraction'].std():.6f}")
print(f"Normalization factor (b_mean): {b_mean:.2f}")

# Verify chamber counts
print("\n" + "=" * 80)
print("CHAMBER VERIFICATION")
print("=" * 80)
for rep_name, n_chambers in total_chambers_per_replicate.items():
    actual = regrowth_per_chamber_df[regrowth_per_chamber_df['replicate'] == rep_name]['position'].nunique()
    status = "✓" if actual == n_chambers else "⚠️"
    print(f"{rep_name}: Expected {n_chambers} chambers, saved {actual} chambers {status}")
print("=" * 80)

Normalization factor (b_mean): 159.66

rep1: 10 chambers analyzed
  Positions: pos1, pos10, pos2, pos3, pos4, pos5, pos6, pos7, pos8, pos9
--------------------------------------------------------------------------------
  rep1: 98 colonies after filtering

✅ Saved merged colonies to: 1_first_colonies_ab_gradient_merged.csv
✅ Saved per-chamber regrowth fractions to: 1_regrowth_fractions_ab_gradient_per_chamber.csv
✅ Saved per-replicate regrowth fractions to: 1_regrowth_fractions_ab_gradient_per_replicate.csv
✅ Saved final regrowth fractions to: 1_regrowth_fractions_ab_gradient_merged.csv

REGROWTH FRACTION PER BIN (mean ± std across replicates)

X-AXIS:
--------------------------------------------------------------------------------
  Bin   0.0-  5.0 µm: 0.004384 ± 0.000000 (n=1)
  Bin   5.0- 10.0 µm: 0.006263 ± 0.000000 (n=1)
  Bin  10.0- 15.0 µm: 0.008769 ± 0.000000 (n=1)
  Bin  15.0- 20.0 µm: 0.006263 ± 0.000000 (n=1)
  Bin  20.0- 25.0 µm: 0.005637 ± 0.000000 (n=1)
  Bin  25.0- 30.0 